# Amazon Bedrock AgentCore Gateway - Tutorial de busca semântica

### Detalhes do Tutorial


| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial    | Conversacional                                                                   |
| Tipo de agente      | Único                                                                            |
| Serviços AgentCore  | AgentCore Gateway, AgentCore Identity                                            |
| Framework de agente | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                        |
| Componentes do tutorial | Criação e uso de AgentCore Gateway com Lambda a partir do Strands Agent       |
| Vertical do tutorial| Cross-vertical                                                                   |
| Complexidade do exemplo | Fácil                                                                        |
| SDK utilizado        | Amazon BedrockAgentCore Python SDK e boto3  

### Arquitetura do Tutorial
O Amazon Bedrock AgentCore Gateway fornece conectividade unificada entre agentes e as ferramentas e recursos com os quais eles precisam interagir. O Gateway desempenha múltiplos papéis nessa camada de conectividade:

1. **Guarda de Segurança**: O Gateway gerencia a autorização OAuth para garantir que apenas usuários/agentes válidos acessem ferramentas/recursos.
2. **Tradutor**: O Gateway traduz as requisições dos agentes feitas usando protocolos populares como o Model Context Protocol (MCP) em requisições de API e invocações Lambda. Isso significa que os desenvolvedores não precisam hospedar servidores, gerenciar integração de protocolos, suporte a versões, correções de versão, etc.
3. **Compositor**: O Gateway permite que os desenvolvedores combinem perfeitamente múltiplas APIs, funções e ferramentas em um único endpoint MCP que um agente pode usar.
4. **Chaveiro**: O Gateway lida com a injeção das credenciais corretas para usar com a ferramenta certa, garantindo que os agentes possam usar perfeitamente ferramentas que requerem diferentes conjuntos de credenciais.
5. **Pesquisador**: O Gateway permite que os agentes pesquisem em todas as suas ferramentas para encontrar apenas as mais adequadas para um determinado contexto ou pergunta. Isso permite que os agentes usem milhares de ferramentas em vez de apenas algumas. Também minimiza o conjunto de ferramentas que precisam ser fornecidas no prompt LLM do agente, reduzindo latência e custo.
6. **Gerenciador de Infraestrutura**: O Gateway é completamente serverless e vem com observabilidade e auditoria integradas, eliminando a necessidade de os desenvolvedores gerenciarem infraestrutura adicional para integrar seus agentes e ferramentas.

![Como funciona](images/gw-arch-overview.png)

### Principais Funcionalidades do Tutorial

* Criação de Amazon Bedrock AgentCore Gateways com alvos baseados em AWS Lambda
* Uso da busca semântica do AgentCore Gateway
* Uso do Strands Agents para mostrar como a busca do AgentCore Gateway melhora a latência

## Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Strands Agents

## O AgentCore Gateway ajuda a resolver o desafio de servidores MCP com grande número de ferramentas
Em um cenário empresarial típico, construtores de agentes encontram servidores MCP com centenas ou até milhares
de ferramentas MCP. Esse volume de ferramentas apresenta desafios para agentes de IA, incluindo baixa precisão na seleção de ferramentas,
aumento de custo e maior latência causada pelo maior uso de tokens devido ao excesso de metadados de ferramentas.
Isso pode acontecer ao conectar seus agentes a serviços de terceiros (ex.: Zendesk, Salesforce,
Slack, JIRA, ...), ou a serviços REST empresariais existentes.
O AgentCore Gateway fornece uma busca semântica integrada entre ferramentas,
que melhora a latência, o custo e a precisão do agente, enquanto ainda fornece aos agentes as ferramentas de que precisam.
Dependendo do seu caso de uso, modelo LLM e framework de agente, você pode obter até 3x melhor latência mantendo
o agente focado em ferramentas relevantes versus fornecer o conjunto completo de centenas de ferramentas de um servidor MCP típico.

![Como funciona](images/gateway_tool_search.png)

## O que você aprenderá neste notebook
Neste notebook, fornecemos um tutorial para a busca do AgentCore Gateway. Ao final deste tutorial passo a passo, você
entenderá:

- Como usar a ferramenta de busca integrada do AgentCore Gateway para encontrar rapidamente ferramentas relevantes
- Como integrar resultados de busca de ferramentas no Strands Agents para melhor latência e custo reduzido

## Visão geral da estrutura do notebook
O notebook está estruturado com as seguintes seções:

1. Entendendo os fundamentos da Busca do AgentCore Gateway
2. Preparando o ambiente do notebook
3. Configurando um Gateway com centenas de ferramentas
4. Pesquisando ferramentas em um Gateway
5. Usando Strands Agents com um servidor MCP que tem muitas ferramentas
6. Adicionando resultados de busca de ferramentas a um Strands Agent
7. Mostrando melhoria de 3x na latência usando busca de ferramentas

# Entendendo os fundamentos da Busca do AgentCore Gateway

Quando você cria um AgentCore Gateway, você tem a opção de indicar que deseja a Busca habilitada.
Para Gateways com busca habilitada, três coisas acontecem:

1. **O vector store é criado**. O serviço Gateway cria automaticamente um vector store serverless totalmente gerenciado para o seu novo Gateway. Isso permite uma busca semântica completa em todas as ferramentas do seu Gateway.
3. **O vector store é populado**. Conforme você adiciona Gateway Targets ao seu Gateway, o serviço usa automaticamente embeddings nos bastidores para popular o vector store com base nas ferramentas do novo Target. Os metadados das ferramentas vêm das definições JSON das suas ferramentas ou da especificação OpenAPI Schema para seus alvos de serviços REST.
2. **A ferramenta de busca (baseada em MCP) é fornecida**. Além de todas as suas ferramentas definidas pelo usuário (de alvos AWS Lambda ou serviços REST), o Gateway recebe uma ferramenta MCP adicional que fornece busca semântica. Ela se chama `x-amz-bedrock-agentcore-search`. O prefixo garante que não haja conflitos de nomes com suas ferramentas definidas pelo usuário. Podemos adicionar mais ferramentas assim no futuro também. A ferramenta de busca tem um único argumento chamado `query`. Quando a ferramenta de busca é invocada, o serviço Gateway realiza uma busca semântica usando essa consulta, comparando-a com os metadados das ferramentas disponíveis (nomes, descrições, esquema de entrada e saída), e retorna as ferramentas mais relevantes em ordem decrescente de relevância.

# Preparando o ambiente do notebook

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

Importe todas as bibliotecas Python necessárias e carregue as variáveis de ambiente

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.handlers import null_callback_handler

from strands.tools.mcp.mcp_client import MCPClient, MCPAgentTool

from mcp.client.streamable_http import streamablehttp_client
from mcp.types import Tool as MCPTool

import logging
import time
import json
import boto3
import requests
import utils

GATEWAY_NAME = "gateway-search-tutorial"

Configure um logger

In [ ]:
# Configure the root strands logger
logging.getLogger("strands").setLevel(logging.ERROR)  # INFO) #DEBUG) #

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()]
)

Verifique nossa versão do boto3

In [ ]:
boto3.__version__

Obtenha nosso cliente boto3 para a API do plano de controle do AgentCore.

In [ ]:
session = boto3.Session()
agentcore_client = session.client(
    "bedrock-agentcore-control",
)

# Configurando um Gateway com centenas de ferramentas

O AgentCore Gateway fornece uma maneira segura e escalável de expor um conjunto curado de APIs existentes
como ferramentas MCP para seus agentes. Em um ambiente de produção, seus recursos do Gateway seriam criados
usando infraestrutura como código com ferramentas como CloudFormation, CDK ou Terraform. Neste tutorial,
usamos as APIs do plano de controle do boto3 diretamente, para que você possa entender os recursos e APIs de forma mais eficaz.
Isso ajudará você a começar mais facilmente a construir e usar seus próprios gateways, e criar agentes mais poderosos
e seguros.

Em alto nível, os passos para configurar seu Gateway são:

1. Definir quais provedores de identidade e provedores de credenciais você está usando para segurança de entrada (agentes chamando Gateways) e saída (Gateways chamando ferramentas).
2. Criar o Gateway usando `create_gateway`.
3. Adicionar Gateway Targets usando `create_gateway_target`, para expor ferramentas MCP que serão implementadas em AWS Lambda ou em serviços RESTful existentes.

Neste tutorial, usaremos o Amazon Cognito como provedor de identidade (IdP), funções AWS Lambda como alvos e AWS IAM para autenticação de saída. Os mesmos conceitos demonstrados neste tutorial ainda se aplicam ao usar outros IdPs ou outros tipos de alvos.

### Criando recursos do Amazon Cognito

Neste tutorial, assumimos que você já criou os seguintes recursos e configurou as variáveis de ambiente correspondentes:

- Função IAM para execução do AWS Lambda (`gateway_lambda_iam_role`)
- Função AWS Lambda para suas ferramentas matemáticas simples (`calc_lambda_arn`)
- Função AWS Lambda para sua ferramenta de reserva de restaurante (`restaurant_lambda_arn`)
- User pools do Amazon Cognito, fornecendo um client id (`cognito_client_id`) e uma URL de descoberta (`cognito_discovery_url`)

Vamos dar uma olhada nos metadados JSON das ferramentas para a API do restaurante. Note que se estivéssemos integrando com serviços REST existentes, as especificações da API seriam fornecidas usando OpenAPI Schema.

In [ ]:
with open("./restaurant/restaurant-api.json") as f:
    data = json.load(f)
print(json.dumps(data, indent=4))

Aqui estão as APIs simples de calculadora.

In [ ]:
with open("./calc/calc-api.json") as f:
    data = json.load(f)[0:3]
print(json.dumps(data, indent=4))

Aqui está a implementação da função AWS Lambda para as ferramentas de calculadora.

In [ ]:
from IPython.display import display, Code

with open("./calc/lambda_function_code.py", "r") as f:
    code_content = f.read()
display(Code(code_content, language="python"))

In [ ]:
with open("./restaurant/lambda_function_code.py", "r") as f:
    code_content = f.read()
display(Code(code_content, language="python"))

In [ ]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
calc_lambda_resp = utils.create_gateway_lambda(
    "calc/lambda_function_code.zip", lambda_function_name="calc_lambda_gateway"
)

if calc_lambda_resp is not None:
    if calc_lambda_resp["exit_code"] == 0:
        print(
            "Lambda function created with ARN: ",
            calc_lambda_resp["lambda_function_arn"],
        )
    else:
        print(
            "Lambda function creation failed with message: ",
            calc_lambda_resp["lambda_function_arn"],
        )

In [ ]:
calc_lambda_resp["lambda_function_arn"]

In [ ]:
#### Create a sample AWS Lambda function that you want to convert into MCP tools
restaurant_lambda_resp = utils.create_gateway_lambda(
    "restaurant/lambda_function_code.zip",
    lambda_function_name="restaurant_lambda_gateway",
)

if restaurant_lambda_resp is not None:
    if restaurant_lambda_resp["exit_code"] == 0:
        print(
            "Lambda function created with ARN: ",
            restaurant_lambda_resp["lambda_function_arn"],
        )
    else:
        print(
            "Lambda function creation failed with message: ",
            restaurant_lambda_resp["lambda_function_arn"],
        )

In [ ]:
restaurant_lambda_resp["lambda_function_arn"]

In [ ]:
cognito_response = utils.setup_cognito_user_pool()

In [ ]:
bearer_token = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)

In [ ]:
gateway_role_arn = utils.create_gateway_iam_role(
    lambda_arns=[
        calc_lambda_resp["lambda_function_arn"],
        restaurant_lambda_resp["lambda_function_arn"],
    ]
)

#### Vamos criar algumas funções auxiliares para usar as APIs do plano de controle

In [ ]:
def read_apispec(json_file_path):
    try:
        # read json file and return contents as string
        with open(json_file_path, "r") as file:
            # Parse JSON to Python object
            api_spec = json.load(file)
            return api_spec

    except FileNotFoundError:
        return f"Error: File {json_file_path} not found"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


def list_gateways():
    response = agentcore_client.list_gateways()
    print(json.dumps(response, indent=2, default=str))
    return response

#### Função auxiliar para criar Gateway
Aqui está uma função auxiliar para criar um AgentCore Gateway dado um nome e
descrição. Ela usa o Amazon Cognito como seu IdP e obtém o client ID permitido
e a URL de descoberta das variáveis de ambiente, pois já foram definidas.
Ela também habilita por padrão a busca semântica no Gateway resultante e usa
uma função IAM predefinida.

In [ ]:
def create_gateway(gateway_name, gateway_desc):
    # Use Cognito for Inbound OAuth to our Gateway
    auth_config = {
        "customJWTAuthorizer": {
            "allowedClients": [cognito_response["client_id"]],
            "discoveryUrl": cognito_response["discovery_url"],
        }
    }
    # Enable semantic search of tools
    search_config = {
        "mcp": {"searchType": "SEMANTIC", "supportedVersions": ["2025-03-26"]}
    }
    # Create the gateway
    response = agentcore_client.create_gateway(
        name=gateway_name,
        roleArn=gateway_role_arn,
        authorizerType="CUSTOM_JWT",
        description=gateway_desc,
        protocolType="MCP",
        authorizerConfiguration=auth_config,
        protocolConfiguration=search_config,
    )
    # print(json.dumps(response, indent=2, default=str))
    return response["gatewayId"]

#### Função auxiliar para criar Gateway Target
Esta função cria um novo alvo AWS Lambda em um Gateway existente.
Basta fornecer o ID do gateway, o nome e a descrição do novo alvo,
o ARN da função AWS Lambda existente e o esquema JSON descrevendo
as interfaces das ferramentas que você deseja expor pelo gateway.

In [ ]:
def create_gatewaytarget(gateway_id, target_name, target_descr, lambda_arn, api_spec):
    # Add a Lambda target to the gateway
    response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=target_name,
        description=target_descr,
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": lambda_arn,
                    "toolSchema": {"inlinePayload": api_spec},
                }
            }
        },
        # Use IAM as credential provider
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    return response["targetId"]

### Criando seu primeiro AgentCore Gateway
Antes de configurar seu primeiro Gateway, vamos dar uma olhada rápida em como
o Gateway fornece segurança, tanto para requisições de entrada para usar ferramentas MCP,
quanto para acesso de saída do Gateway para ferramentas e recursos.

![Como funciona](images/gateway_secure_access.png)

Agora vamos criar o gateway para este tutorial, fornecendo um nome e uma descrição.

In [ ]:
print(f"Create gateway with name: {GATEWAY_NAME}")
gatewayId = create_gateway(
    gateway_name=GATEWAY_NAME, gateway_desc="AgentCore Gateway Tutorial"
)
print(f"Gateway created with id: {gatewayId}.")

### Adicionando Gateway Targets do AgentCore
Neste tutorial, assumimos que você já instalou um par de funções Lambda, uma para fazer cálculos
matemáticos simples e outra que simula a criação de uma reserva de restaurante. Vamos adicionar um Gateway Target
para cada uma dessas funções.

Depois de adicionar esses alvos, adicionaremos alvos extras para simplesmente aumentar a contagem de ferramentas MCP,
ajudando a demonstrar o poder da busca do AgentCore Gateway.

Agora que o gateway foi criado, vamos adicionar um alvo para fazer reservas de restaurante
via uma função Lambda.

In [ ]:
restaurant_api_spec = read_apispec("./restaurant/restaurant-api.json")
restaurant_lambda_arn = restaurant_lambda_resp["lambda_function_arn"]
print(f"Restaurant Lambda ARN: {restaurant_lambda_arn}")

restaurantTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=restaurant_lambda_arn,
    target_name="FoodTools",
    target_descr="Restaurant Tools",
    api_spec=restaurant_api_spec,
)
print(f"RestaurantTarget created with id: {restaurantTargetId} on gateway: {gatewayId}")

Aqui vamos adicionar um segundo alvo, desta vez com uma Lambda que implementa 4 ferramentas básicas (adicionar, subtrair,
multiplicar, dividir), e um conjunto de 75 definições de ferramentas geradas para gestão de investimentos (negociação, pesquisa de crédito, análise quantitativa, gestão de portfólio). As definições de ferramentas de gestão de investimentos não são
realmente implementadas na função Lambda. Estamos apenas adicionando-as para demonstrar um grande volume de ferramentas.

In [ ]:
calc_api_spec = read_apispec("./calc/calc-api.json")
print(f"API spec for calc has {len(calc_api_spec)} functions\n")
calc_lambda_arn = calc_lambda_resp["lambda_function_arn"]
print(f"Calc Lambda ARN: {calc_lambda_arn}")

time.sleep(5)
calcTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=calc_lambda_arn,
    target_name="CalcTools",
    target_descr="Calculation Tools",
    api_spec=calc_api_spec,
)
print(f"CalcTools Target created with id: {calcTargetId} on gateway: {gatewayId}")

Para demonstrar o poder da busca do gateway, agora adicionamos mais algumas cópias do alvo Calculadora,
para que terminemos com mais de 300 ferramentas MCP expostas.

In [ ]:
def add_more_tools(gatewayId):
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc2",
        target_descr="Calculation 2 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc2 Target created with id: {calcTargetId} on gateway: {gatewayId}")
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc3",
        target_descr="Calculation 3 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc3 Target created with id: {calcTargetId} on gateway: {gatewayId}")
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc4",
        target_descr="Calculation 4 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc4 Target created with id: {calcTargetId} on gateway: {gatewayId}")

In [ ]:
add_more_tools(gatewayId=gatewayId)

In [ ]:
resp = agentcore_client.list_gateway_targets(gatewayIdentifier=gatewayId)
targets = resp["items"]
for target in resp["items"]:
    print(f"{target['name']} - {target['description']}")

# Pesquisando ferramentas em um Gateway

### Familiarizando-se com a listagem de ferramentas MCP antes de pesquisar

Vamos definir algumas funções utilitárias para recuperar a URL do endpoint MCP para um determinado ID de Gateway, e para
recuperar nosso token de acesso JWT OAuth para usar nosso Gateway de forma segura.

In [ ]:
def get_gateway_endpoint(gateway_id):
    response = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = response["gatewayUrl"]
    return gateway_url

Agora que nosso Gateway está criado e tem alvos, vamos obter a URL MCP desse
Gateway. Podemos recuperar a URL do endpoint a partir do plano de controle do Gateway com base no ID do Gateway.

#### Usando o MCP Inspector com seu Gateway

Agora que temos uma URL de endpoint para o servidor MCP e um token JWT bearer, você pode querer explorar
o servidor MCP com a ferramenta MCP Inspector. O MCP Inspector é uma ferramenta de código aberto que pode se conectar a qualquer servidor
MCP, permite listar as ferramentas fornecidas e até oferece uma experiência fácil de invocação de ferramentas.

Na sua janela de terminal, simplesmente digite `npx @modelcontextprotocol/inspector` para iniciar o MCP Inspector. Em seguida, cole
a URL do endpoint do seu Gateway e seu token JWT para conectar. Uma vez conectado, experimente Listar Ferramentas e Invocar Ferramenta.

Aqui está uma captura de tela de exemplo.

![MCP Inspector](images/mcp_inspector.png)

In [ ]:
gatewayEndpoint = get_gateway_endpoint(gateway_id=gatewayId)
print(f"Gateway Endpoint - MCP URL: {gatewayEndpoint}")

A segurança do servidor MCP é baseada em OAuth. Para interagir com nosso Gateway, precisaremos
recuperar um token de acesso JWT OAuth do nosso IdP.

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
print(f"Bearer token: {jwtToken}")

In [ ]:
# !npx @modelcontextprotocol/inspector

#### Criando funções auxiliares que usam jsonrpc para invocar ferramentas MCP ou listá-las
Vamos definir uma função auxiliar chamada `invoke_gateway_tool` que usa jsonrpc para invocar qualquer uma das
ferramentas MCP expostas por um servidor MCP, incluindo, é claro, seu Gateway. Dado uma URL de endpoint e um token JWT,
você pode usar este utilitário para invocar qualquer uma das ferramentas MCP que o AgentCore Gateway disponibilizou
para você quando você adicionou Gateway Targets ao seu Gateway.

In [ ]:
def invoke_gateway_tool(gateway_endpoint, jwt_token, tool_params):
    # print(f"Invoking tool {tool_params['name']}")

    requestBody = {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": tool_params,
    }
    response = requests.post(
        gateway_endpoint,
        json=requestBody,
        headers={
            "Authorization": f"Bearer {jwt_token}",
            "Content-Type": "application/json",
        },
    )

    return response.json()

Aqui está outra função utilitária para usar o método `tools/list` do MCP para listar as ferramentas MCP
disponíveis no seu Gateway. Dado um ID de Gateway e um Token JWT, ela recupera o conjunto completo
de ferramentas desse Gateway e retorna uma lista em formato pronto para o agente. A lista retornada contém
objetos MCPAgentTool do Strands Agents que são adequados para entregar ao seu Agente.

Note que a chamada `tools/list` é paginada, então a função precisa fazer um loop, obtendo uma página de
ferramentas por vez, até que o campo `nextCursor` não esteja mais preenchido. A função utilitária chama diretamente
o endpoint usando HTTPS e o protocolo jsonrpc. Esta é uma forma de nível mais baixo para listar ferramentas
comparada à classe `MCPClient` fornecida pelo Strands Agents. Veremos essa experiência mais adiante.

In [ ]:
def get_all_agent_tools_from_mcp_endpoint(gateway_endpoint, jwt_token, client):
    more_tools = True
    tools_count = 0
    tools_list = []

    requestBody = {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
    next_cursor = ""

    while more_tools:
        if tools_count == 0:
            requestBody["params"] = {}
        else:
            print(f"\nGetting next page of tools since a next cursor was returned\n")
            requestBody["params"] = {"cursor": next_cursor}

        headers = {
            "Authorization": f"Bearer {jwt_token}",
            "Content-Type": "application/json",
        }

        print(f"\n\nListing tools for gateway {gateway_endpoint}")

        response = requests.post(gateway_endpoint, json=requestBody, headers=headers)

        tools_json = response.json()
        tools_count += len(tools_json["result"]["tools"])

        for tool in tools_json["result"]["tools"]:
            mcp_tool = MCPTool(
                name=tool["name"],
                description=tool["description"],
                inputSchema=tool["inputSchema"],
            )
            mcp_agent_tool = MCPAgentTool(mcp_tool, client)
            short_descr = tool["description"][0:40] + "..."
            print(f"adding tool '{mcp_agent_tool.tool_name}' - {short_descr}")
            tools_list.append(mcp_agent_tool)

        if "nextCursor" in tools_json["result"]:
            next_cursor = tools_json["result"]["nextCursor"]
            more_tools = True
        else:
            more_tools = False

    print(f"\nTotal tools found: {tools_count}\n")
    return tools_list

Vamos usar esta função auxiliar e ver os resultados.

In [ ]:
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    all_tools = get_all_agent_tools_from_mcp_endpoint(
        gateway_endpoint=gatewayEndpoint, jwt_token=jwtToken, client=client
    )
    print(f"\nFound {len(all_tools)} tools using jsonrpc to list MCP tools\n")

#### Usando list_tools_sync() do Strands Agents com paginação
Se você já escreveu algum cliente MCP baseado em Python, provavelmente está familiarizado com o método `list_tools_sync()`
que retorna o conjunto de ferramentas disponíveis no servidor MCP ao qual o cliente está associado.
Mas você sabia que a listagem de ferramentas MCP também é paginada? Por padrão, você receberá apenas o primeiro pequeno
subconjunto de ferramentas retornado. Para servidores MCP simples, você pode não ter percebido isso, mas para muitos servidores
MCP do mundo real, seu código precisa fazer um loop, pegando páginas de ferramentas por vez
até que não haja mais páginas restantes. O utilitário `get_all_mcp_tools_from_mcp_client` faz exatamente isso.
Ele retorna a lista completa de ferramentas de um determinado Strands Agent MCP Client.

In [ ]:
def get_all_mcp_tools_from_mcp_client(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True
            pagination_token = tmp_tools.pagination_token
    return tools

Vamos experimentar com nosso Gateway e descobrir quantas ferramentas o cliente Python encontra.
Primeiro criamos um objeto MCPClient baseado na URL do nosso endpoint e no nosso token JWT bearer. Em seguida,
recuperamos o conjunto completo de ferramentas através de muitas páginas de ferramentas retornadas pelo servidor MCP.
Dados os alvos que adicionamos anteriormente, isso deve retornar mais de 300 ferramentas.

In [ ]:
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

Agora vimos 3 maneiras diferentes de obter o conjunto completo de ferramentas do seu Gateway usando-o
como um servidor MCP:

1. diretamente usando jsonrpc
2. usando o método `list_tools_sync()` no MCPClient do Strands Agent
3. usando a ferramenta MCP Inspector (que usa jsonrpc internamente).

Para desenvolvedores típicos construindo agentes, você usará a opção 2.

### Usando a ferramenta de busca semântica integrada do Gateway
Agora vamos experimentar nossa primeira busca semântica no Gateway usando sua ferramenta de busca integrada fornecida como
uma ferramenta MCP adicional que é adicionada à sua lista de ferramentas MCP.

Primeiro, vamos definir uma função utilitária simples para executar a ferramenta de busca usando MCP.
Assim como para listar ferramentas, precisamos do endpoint do gateway e do token JWT. Além disso,
tudo o que precisamos passar é a consulta de busca. A ferramenta de busca do Gateway fará o resto,
comparando essa consulta com o vector store serverless que ele gerencia automaticamente em seu nome.

In [ ]:
def tool_search(gateway_endpoint, jwt_token, query):
    toolParams = {
        "name": "x_amz_bedrock_agentcore_search",
        "arguments": {"query": query},
    }
    toolResp = invoke_gateway_tool(
        gateway_endpoint=gateway_endpoint, jwt_token=jwt_token, tool_params=toolParams
    )
    tools = toolResp["result"]["structuredContent"]["tools"]
    return tools

In [ ]:
start_time = time.time()
tools_found = tool_search(
    gateway_endpoint=gatewayEndpoint,
    jwt_token=jwtToken,
    query="find me 3 credit research tools",
)
end_time = time.time()
print(
    f"tool search via direct Gateway invocation took {(end_time - start_time):.2f} seconds"
)
print(f"Top tool: {tools_found[0]['name']}")

Observe como a busca retorna rápido, em menos de um segundo na maioria dos casos. Os resultados são retornados
em ordem decrescente de relevância da busca com base na correspondência da consulta com os metadados da ferramenta.
As ferramentas mais relevantes estão primeiro na lista. A implementação inicial da busca retorna
até 10 resultados. Você pode então usar todas essas ferramentas no seu agente, ou simplesmente escolher um subconjunto das
correspondências mais relevantes.

# Usando Strands Agents com um servidor MCP que tem muitas ferramentas

Primeiro, selecionamos um modelo para usar com nosso Strands Agent.
Para este notebook, estamos usando modelos Amazon Bedrock, mas o Strands e o AgentCore
podem trabalhar com qualquer LLM.

In [ ]:
bedrockmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.7,
    streaming=True,
    boto_session=session,
)

#### Strands Agent simples usando AgentCore Gateway para ferramentas do agente
Agora vamos mostrar como é fácil usar um Strands Agent para aproveitar um servidor MCP
fornecido pelo AgentCore Gateway. No nosso
exemplo simples, pedimos ao agente para somar alguns números.

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

    simple_agent = Agent(
        model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler
    )
    result = simple_agent("add 100 plus 50 pass ")
    print(f"{result.message['content'][0]['text']}")

O framework Strands Agents também permite que você ignore o loop de eventos do agente, invocando uma ferramenta MCP diretamente.
Como as ferramentas do Gateway são expostas como ferramentas MCP nativas, isso também pode ser feito com ferramentas do Gateway. Aqui
chamamos uma ferramenta MCP do Gateway usando a sintaxe `agent.tool.<nome_ferramenta>(args)`:

```python
direct_result = simple_agent.tool.Calc2___add_numbers(firstNumber=10, secondNumber=20)
resp_json = json.loads(direct_result['content'][0]['text'])
```

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

    simple_agent = Agent(
        model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler
    )
    direct_result = simple_agent.tool.Calc2___add_numbers(
        firstNumber=10, secondNumber=20
    )
    print(f"direct result = {direct_result}")

In [ ]:
def get_search_tool(client):
    mcp_tool = MCPTool(
        name="x_amz_bedrock_agentcore_search",
        description="A special tool that returns a trimmed down list of tools given a context. Use this tool only when there are many tools available and you want to get a subset that matches the provided context.",
        inputSchema={
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "search query to use for finding tools",
                }
            },
            "required": ["query"],
        },
    )
    return MCPAgentTool(mcp_tool, client)

In [ ]:
def search_using_strands(client, query):
    simple_agent = Agent(
        model=bedrockmodel,
        tools=[get_search_tool(client)],
        callback_handler=null_callback_handler,
    )

    direct_result = simple_agent.tool.x_amz_bedrock_agentcore_search(query=query)

    resp_json = json.loads(direct_result["content"][0]["text"])
    search_results = resp_json["tools"]
    # print(json.dumps(search_results, indent=4))
    return search_results

In [ ]:
def find_strands_tools(client, query, top_n):
    strands_mcp_tools = []
    results = search_using_strands(client, query)
    for tool in results[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        strands_mcp_tools.append(MCPAgentTool(mcp_tool, client))
    return strands_mcp_tools

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    simple_agent = Agent(
        model=bedrockmodel,
        tools=[get_search_tool(client)],
        callback_handler=null_callback_handler,
    )

    direct_result = simple_agent.tool.x_amz_bedrock_agentcore_search(
        query="find equity trading tools"
    )

    resp_json = json.loads(direct_result["content"][0]["text"])
    search_results = resp_json["tools"]
    print(json.dumps(search_results, indent=4))

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    results = search_using_strands(client, "find trading tools")
    print(json.dumps(search_results[0], indent=4))

    results = search_using_strands(client, "find credit research tools")
    print(json.dumps(search_results[0], indent=4))

# Adicionando resultados de busca de ferramentas a um Strands Agent

Agora vamos ver como as ferramentas retornadas de uma busca podem ser adicionadas a um
Strands Agent. Para simplificar a codificação, vamos fornecer uma função utilitária que
mapeia resultados de busca de ferramentas para objetos MCPAgentTool do Strands. Basta passar os
resultados da busca e indicar quantos desses resultados você deseja passar para o seu
agente.

In [ ]:
def tools_to_strands_mcp_tools(tools, top_n):
    strands_mcp_tools = []
    for tool in tools[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        strands_mcp_tools.append(MCPAgentTool(mcp_tool, client))
    return strands_mcp_tools

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    agent = Agent(
        model=bedrockmodel,
        tools=find_strands_tools(
            client,
            "tools for doing addition, subtraction, multiplication, division",
            10,
        ),
    )
    result = agent("(10*2)/(5-3)")
    print(f"{result.message['content'][0]['text']}")

In [ ]:
%%time

jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)
with client:
    print("Searching for an ADDING tool from endpoint with full set of tools...")
    tools_found = tool_search(
        gateway_endpoint=gatewayEndpoint,
        jwt_token=jwtToken,
        query="tools for multiplying two numbers",
    )
    print(f"Top tool found: {tools_found[0]['name']}\n")

    agent = Agent(model=bedrockmodel, tools=tools_to_strands_mcp_tools(tools_found, 1))
    result = agent("10 * 70")
    print(f"{result.message['content'][0]['text']}")

Observe a melhoria de latência. Este exemplo usando um subconjunto de ferramentas da busca do Gateway é significativamente mais rápido do que
a invocação do agente dependendo de centenas de ferramentas.

# Mostrando melhoria de 3x na latência usando busca de ferramentas

Agora que sabemos como usar ferramentas MCP do Gateway a partir de um Strands Agent, e sabemos como pesquisar ferramentas e
adicioná-las a um agente, vamos mostrar o poder da busca. Destacaremos a redução significativa de latência
e o uso de tokens de entrada que podem ser alcançados.

Para demonstrar as reduções de latência e tokens, comparamos 2 abordagens lado a lado:

1. **Sem busca**. Adicionamos o conjunto completo de ferramentas MCP que o servidor MCP expõe (mais de 300 no nosso caso) ao nosso agente e deixamos o agente fazer sua seleção e invocação de ferramentas adequadamente.
2. **Usando busca**. Na segunda abordagem, fazemos uma busca baseada no tópico em questão e enviamos apenas as ferramentas mais relevantes para o agente. Para provar o ponto, usamos dois tópicos diferentes: matemática (somar números) e comida (reservar uma mesa em restaurante), cada um requerendo um conjunto diferente de ferramentas.

Para normalizar a distribuição de latência e obter uma comparação significativa, realizamos múltiplas iterações de
cada abordagem. Além disso, para evitar superestimar os ganhos, ao usar a abordagem de busca, incluímos não apenas a
latência da invocação do agente, mas também a latência de realizar a busca de ferramentas. Para cada
iteração, entregamos ao agente duas tarefas:

1. Tarefa de matemática - somar 2 números
2. Tarefa de comida - reservar uma mesa em restaurante

Os resultados abaixo demonstram os benefícios, destacando redução de 3x na latência e uma redução ainda maior no
uso de tokens de entrada. Note que embora a economia de uso de tokens se traduza em economia de custos, isso pode não ser tão impactante devido ao
custo relativamente menor dos tokens de entrada (para muitos provedores de modelos, tokens de entrada são muito menos custosos). Mesmo assim, para
implantações de agentes em larga escala, até os custos de tokens de entrada podem se acumular, então a busca dinâmica pode ajudar a reduzir
os custos de execução do agente também.

#### Medir latência e uso de tokens para agente usando o conjunto completo de ferramentas MCP

In [ ]:
iterations = 2
full_tokens = light_tokens = 0
full_elapsed_time = light_elapsed_time = 0

jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(
    lambda: streamablehttp_client(
        f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}
    )
)

In [ ]:
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")
    heavy_agent = Agent(
        model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler
    )

    math_input = "add 100 plus <iteration>"
    food_input = (
        "book me a table for 2 at Burger King under name Jo Smith at 7pm August <day>"
    )

    print("using agent with ALL tools...")
    start_time = time.time()

    for i in range(iterations):
        result = heavy_agent(math_input.replace("<iteration>", str(i + 1)))
        print(f"{i+1}) {result.message['content'][0]['text']}")

        result = heavy_agent(food_input.replace("<day>", str(i + 1)))
        print(f"{i+1}) {result.message['content'][0]['text']}")

    end_time = time.time()
    full_tokens = result.metrics.accumulated_usage["totalTokens"]
    full_elapsed_time = end_time - start_time
    print(f"\nTotal time: {full_elapsed_time:.1f} s, tokens: {full_tokens:,d}\n")

#### Medir latência e uso de tokens para agentes usando a Busca do Gateway
Agora usaremos uma abordagem dinâmica, chamando a busca para encontrar ferramentas relevantes e depois chamando o
agente apenas com essas ferramentas relevantes. Note que como estamos reiniciando o agente a cada
turno de conversa, também estamos inicializando a lista de mensagens a partir do histórico de conversa do turno anterior.

In [ ]:
with client:
    print("using agent with ONLY tools from focused search...")
    start_time = time.time()
    messages = []

    light_agent = Agent()

    for i in range(iterations):
        print("Searching for an ADDING tool from endpoint with full set of tools...")
        tools_found = tool_search(
            gateway_endpoint=gatewayEndpoint,
            jwt_token=jwtToken,
            query="tools for simply adding two numbers",
        )
        print(f"Top tool found: {tools_found[0]['name']}\n")
        light_agent = Agent(
            model=bedrockmodel,
            tools=tools_to_strands_mcp_tools(tools_found, 1),
            messages=messages,
            callback_handler=null_callback_handler,
        )
        light_result = light_agent(math_input.replace("<iteration>", str(i + 1)))
        print(f"{i+1}) {light_result.message['content'][0]['text']}")
        messages = light_agent.messages

        print(
            "Searching for a RESTAURANT BOOKING tool from endpoint with full set of tools..."
        )
        tools_found = tool_search(
            gateway_endpoint=gatewayEndpoint,
            jwt_token=jwtToken,
            query="tools for booking a restaurant reservation",
        )
        print(f"Top tool found: {tools_found[0]['name']}\n")
        light_agent = Agent(
            model=bedrockmodel,
            tools=tools_to_strands_mcp_tools(tools_found, 1),
            messages=messages,
            callback_handler=null_callback_handler,
        )
        light_result = light_agent(food_input.replace("<day>", str(i + 1)))
        print(f"{i+1}) {light_result.message['content'][0]['text']}")
        messages = light_agent.messages
        light_tokens = light_result.metrics.accumulated_usage["totalTokens"]
    end_time = time.time()

    light_elapsed_time = end_time - start_time
    print(f"\nTotal time: {light_elapsed_time:.1f} s, tokens: {light_tokens:,d}\n")

#### Comparar resultados, destacando os benefícios da busca

In [ ]:
print(
    f"\n\nLatency without search: {full_elapsed_time:.1f}s, using search: {light_elapsed_time:.1f}s"
)
print(f"Tokens without search: {full_tokens:,d}, using search: {light_tokens:,d}")

# Conclusão
Neste tutorial, você aprendeu sobre o Amazon Bedrock AgentCore Gateway e sua capacidade de
busca semântica totalmente gerenciada integrada. Você viu o seguinte:

- como criar um gateway com busca semântica habilitada
- como adicionar múltiplos gateway targets para disponibilizar mais de 300 ferramentas MCP a partir de um único endpoint
- como listar as ferramentas no seu gateway usando 3 abordagens diferentes
- como usar a ferramenta de busca semântica integrada para encontrar ferramentas relevantes
- como integrar a busca com seu Strands Agent
- como comparar o desempenho de um agente usando um servidor com centenas de ferramentas versus um que usa busca semântica para restringir ferramentas a um tópico específico

A busca do AgentCore Gateway é útil para casos de uso mais avançados também. Ao oferecer a busca como uma ferramenta
MCP nativa e não apenas uma API de plano de controle, você pode imaginar dar aos seus agentes mais autonomia para descobrir novos
servidores MCP e encontrar novas capacidades em tempo de execução, levando a avanços na resolução de problemas mais desafiadores.
Além disso, a busca é uma base importante para registros MCP e para apoiar desenvolvedores de agentes enquanto
projetam e constroem novos agentes.

# Limpeza de recursos

Primeiro vamos definir algumas funções auxiliares para limpar os recursos do AgentCore Gateway.

In [ ]:
def delete_gatewaytarget(gateway_id):
    response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)

    print(f"Found {len(response['items'])} targets for the gateway")

    for target in response["items"]:
        print(
            f"Deleting target with Name: {target['name']} and Id: {target['targetId']}"
        )

        response = agentcore_client.delete_gateway_target(
            gatewayIdentifier=gateway_id, targetId=target["targetId"]
        )
        time.sleep(20)


def delete_gateway(gateway_id):
    response = agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)

### Excluindo Gateway Targets

In [ ]:
delete_gatewaytarget(gateway_id=gatewayId)

### Excluindo o Gateway

In [ ]:
delete_gateway(gateway_id=gatewayId)

In [ ]:
lambda_arns = [
    calc_lambda_resp["lambda_function_arn"],
    restaurant_lambda_resp["lambda_function_arn"],
]

for arn in lambda_arns:
    if utils.delete_gateway_lambda(arn):
        print(f"Deleted Lambda: {arn}")
    else:
        print(f"Lambda {arn} not found or deletion failed")

In [ ]:
# Gateway role cleanup
if utils.delete_gateway_iam_role():
    print("Gateway IAM role deleted")
else:
    print("Gateway IAM role not found or deletion failed")

# Cognito cleanup
if utils.delete_cognito_user_pool():
    print("Cognito pool deleted")
else:
    print("✗ Failed to delete Cognito pool")